# **PUBLIC PERCEPTIONS OF GOVERNMENT POLICIES TO PUBLIC HEALTH:**
## A comparative study in South Karnataka region

By: Varnica Sharma and Aman Tripathi




In [ ]:
# !pip install praw
import praw
import pandas as pd
import time

# Setup Reddit client
reddit = praw.Reddit(
    client_id='YOUR_CLIENT_ID',
    client_secret='YOUR_CLIENT_SECRET',
    user_agent='YOUR_USER_AGENT'
)

# Subreddits to search
subreddits = ['bangalore', 'karnataka', 'mysore', 'southindia']

# Keywords to filter
keywords = ['health', 'policy', 'government', 'karnataka', 'bengaluru', 'mysuru', 'mandya', 'hassan', 'tumkur', 'chamarajanagar']

# Initialize list for dataset
data = []

# Start scraping
for sub in subreddits:
    print(f"Scraping r/{sub}...")
    subreddit = reddit.subreddit(sub)

    for submission in subreddit.new(limit=300):  # Increase limit if needed
        title = submission.title.lower()
        selftext = submission.selftext.lower()
        url = submission.url
        created_utc = submission.created_utc
        post_region = ''

        # Check if any keyword exists in title or body
        if any(keyword in title or keyword in selftext for keyword in keywords):
            # Guess region based on subreddit or title keywords
            if 'mysore' in sub or 'mysuru' in title:
                post_region = 'Mysuru'
            elif 'bangalore' in sub or 'bengaluru' in title:
                post_region = 'Bengaluru'
            elif 'mandya' in title:
                post_region = 'Mandya'
            elif 'hassan' in title:
                post_region = 'Hassan'
            elif 'tumkur' in title:
                post_region = 'Tumkur'
            elif 'chamarajanagar' in title:
                post_region = 'Chamarajanagar'
            else:
                post_region = 'Other South Karnataka'

            # Fetch top-level comments
            submission.comments.replace_more(limit=0)
            for comment in submission.comments.list():
                comment_body = comment.body.lower()
                if any(keyword in comment_body for keyword in keywords):
                    data.append({
                        'Comment Text': comment.body,
                        'Article Title': submission.title,
                        'Article URL': url,
                        'Article Publication Date': pd.to_datetime(created_utc, unit='s'),
                        'Region': post_region
                    })

        time.sleep(0.5)

# Convert to DataFrame
df = pd.DataFrame(data)

# Save to CSV
df.to_csv('reddit_comments_south_karnataka.csv', index=False)

print(f"Scraped {len(df)} comments.")
print("Saved to 'reddit_comments_south_karnataka.csv'")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
original_df = pd.read_csv("/content/drive/MyDrive/reddit_comments_south_karnataka.csv")
original_df.head()

In [ ]:
original_df.shape

In [ ]:
df = original_df.copy()

In [ ]:
print(df.columns)

# Method 1: Classification
#### **SVM Machine Learning model** (Classification done library NRCLex)

In [ ]:
!pip install nrclex

In [ ]:
# !pip install nrclex
import ssl
import nltk
import textblob.download_corpora as dl

# Fix SSL verification issues (for NLTK downloads)
try:
    _create_unverified_https_context = ssl._create_unverified_context
    ssl._create_default_https_context = _create_unverified_https_context
except AttributeError:
    pass

# Download required corpora
nltk.download('punkt')
dl.download_all()

# Import libraries
import pandas as pd
import spacy
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from textblob import TextBlob
from nrclex import NRCLex


# Preprocessing using spaCy only
nlp = spacy.load("en_core_web_sm")
def preprocess_spacy(text):
    doc = nlp(text.lower())
    tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
    return " ".join(tokens)

df['clean_text'] = df['Comment Text'].astype(str).apply(preprocess_spacy)

# Emotion labeling using NRCLex
def get_dominant_emotion(text):
    emotions = NRCLex(text).raw_emotion_scores
    if emotions:
        return max(emotions, key=emotions.get)
    else:
        return "neutral"

df['emotion_label'] = df['Comment Text'].astype(str).apply(get_dominant_emotion)

# Visualize emotion distribution
df['emotion_label'].value_counts().plot(kind='bar', title='Emotion Distribution', color='skyblue')
plt.xlabel("Emotion")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Vectorize text using TF-IDF
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['clean_text'])
y_emotion = df['emotion_label']

# Train-test split
X_train_em, X_test_em, y_train_em, y_test_em = train_test_split(X, y_emotion, test_size=0.2, random_state=42)

# Train SVM classifier
svm_model = SVC(kernel='linear')
svm_model.fit(X_train_em, y_train_em)
y_pred_em = svm_model.predict(X_test_em)

# Evaluation
print("Emotion Classification Accuracy (SVM):", accuracy_score(y_test_em, y_pred_em))
print("Emotion Classification Report (SVM):\n", classification_report(y_test_em, y_pred_em))


### **Large Language Model (DistilBERT)**

In [ ]:
# !pip install transformers

import pandas as pd
import spacy
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TextClassificationPipeline
import torch


# Preprocessing using spaCy
nlp = spacy.load("en_core_web_sm")
def preprocess_spacy(text):
    doc = nlp(text.lower())
    tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
    return " ".join(tokens)

df['clean_text'] = df['Comment Text'].astype(str).apply(preprocess_spacy)

# Load pre-trained emotion classification model
model_name = "joeddav/distilbert-base-uncased-go-emotions-student"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
pipeline = TextClassificationPipeline(model=model, tokenizer=tokenizer, return_all_scores=True, device=0 if torch.cuda.is_available() else -1)

# Predict emotion labels
def predict_emotion(text):
    scores = pipeline(text[:512])[0]  # Limit to 512 tokens
    scores_sorted = sorted(scores, key=lambda x: x['score'], reverse=True)
    return scores_sorted[0]['label']

df['llm_emotion'] = df['clean_text'].astype(str).apply(predict_emotion)

# Visualize emotion distribution
df['llm_emotion'].value_counts().plot(kind='bar', title='Emotion Distribution (LLM)', color='coral')
plt.xlabel("Emotion")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Evaluate with a dummy train-test split (for comparison structure)
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['llm_emotion'], test_size=0.2, random_state=42
)
y_pred_llm = df.loc[X_test.index, 'llm_emotion']

print("LLM Emotion Classification Report:\n", classification_report(y_test, y_pred_llm))


#Method 2: Clustering
### **SVM Machine Learning Model** (Using TF-IDF + KMeans)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

# Ensure 'Comment Text' column exists
df = df.dropna(subset=['Comment Text'])
texts = df['Comment Text'].astype(str)

# TF-IDF vectorization with bigrams
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_df=0.95, min_df=3)
X_tfidf = vectorizer.fit_transform(texts)

# KMeans clustering
kmeans = KMeans(n_clusters=5, random_state=42)
df['cluster_label'] = kmeans.fit_predict(X_tfidf)

# Visualize cluster distribution
df['cluster_label'].value_counts().sort_index().plot(kind='bar', title='TF-IDF + KMeans Clusters', color='skyblue')
plt.xlabel("Cluster")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# Interpret top terms in each cluster
terms = vectorizer.get_feature_names_out()
for i in range(5):
    center = kmeans.cluster_centers_[i]
    top_terms = center.argsort()[-10:][::-1]
    print(f"\nKMeans Cluster {i} Top Terms: {[terms[j] for j in top_terms]}")

# Train-test split for SVM using cluster labels as pseudo-labels
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, df['cluster_label'], test_size=0.2, random_state=42)
svm_model = SVC(kernel='linear')
svm_model.fit(X_train, y_train)
y_pred = svm_model.predict(X_test)

# Evaluation
print("\nSVM Accuracy on KMeans Labels:", accuracy_score(y_test, y_pred))
print("SVM Classification Report:\n", classification_report(y_test, y_pred))


### **Large Language Model (TopicBERT)**

In [ ]:
!pip install bertopic sentence-transformers

In [ ]:
# !pip install bertopic sentence-transformers

import pandas as pd
import matplotlib.pyplot as plt
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

# Ensure 'Comment Text' column exists
df = df.dropna(subset=["Comment Text"])
texts = df["Comment Text"].astype(str).tolist()

# Vectorizer for BERTopic
vectorizer_model = CountVectorizer(ngram_range=(1, 2), stop_words="english")

# BERTopic modeling
topic_model = BERTopic(language="english", vectorizer_model=vectorizer_model, verbose=True)
topics, probs = topic_model.fit_transform(texts)

# Add topics to dataframe
df['Topic'] = topics

# Display top keywords per topic
print("\nTop 5 Topics with Keywords:")
for i in range(5):
    print(f"\nTopic {i}:")
    print(topic_model.get_topic(i))

# Visualize topic distribution
df['Topic'].value_counts().sort_index().plot(kind='bar', color='mediumseagreen', title='Topic Distribution (BERTopic)')
plt.xlabel("Topic ID")
plt.ylabel("Number of Comments")
plt.tight_layout()
plt.show()

# Train SVM using TF-IDF features and BERTopic topics as labels
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_df=0.95, min_df=3)
X_vec = vectorizer.fit_transform(texts)
X_train, X_test, y_train, y_test = train_test_split(X_vec, df['Topic'], test_size=0.2, random_state=42)

svm = SVC(kernel='linear')
svm.fit(X_train, y_train)
y_pred = svm.predict(X_test)

# Evaluation
print("\nSVM Accuracy on BERTopic Labels:", accuracy_score(y_test, y_pred))
print("SVM Classification Report:\n", classification_report(y_test, y_pred))


In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
clustered_df = df[df['Topic'] != -1]
clustered_df.head()

In [ ]:
clustered_df.shape